# Engagement 1 — Feature Engineering

**Client:** GuardianShield Insurance (fictional)
**Engagement:** Customer Renewal Prediction
**Notebook:** 02 — Feature Engineering
**Author:** Yinka Adebimpe

## Purpose

Transform raw renewal data into a model-ready feature matrix:

1. Drop identifier and leakage columns
2. Encode categorical features
3. Create derived behavioural features
4. Save the prepared matrix to Postgres (`retention.features`) and CSV

In [1]:
# ============================================
# SETUP
# ============================================

import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

from config import DB_URL, DB_SCHEMA, PROCESSED_DATA_DIR

engine = create_engine(DB_URL)

print(f"Project root: {PROJECT_ROOT}")
print(f"Schema: {DB_SCHEMA}")

Project root: C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\OkYeahInsight-Analytics\GuardianShield-Insurance\Engagement-1-Customer-Renewal
Schema: retention


In [3]:
# ============================================
# LOAD RAW RENEWALS
# ============================================

df = pd.read_sql(
    f"SELECT * FROM {DB_SCHEMA}.renewals",
    engine,
    parse_dates=["decision_date"],
)

print(f"Loaded {len(df):,} renewal decisions")
print(f"Columns: {len(df.columns)}")
df.head()

Loaded 12,199 renewal decisions
Columns: 19


,customer_id,policy_id,decision_date,policy_year,tenure_months,annual_premium,premium_change_pct,claims_last_year,total_claims,payment_frequency,policy_type,acquisition_channel,customer_age,region,excess_amount,no_claims_discount,last_year_renewed,segment,will_renew
0,GS-000001,POL-000001-01,2021-05-10,1,12,792.25,0.033,0,0,Monthly,Home,Online,64,Scottish Borders,250.0,0.61,True,standard,1
1,GS-000001,POL-000001-01,2022-05-10,2,24,793.53,0.034,0,0,Monthly,Home,Online,64,Scottish Borders,250.0,0.61,True,standard,1
2,GS-000001,POL-000001-01,2023-05-10,3,36,824.42,0.074,0,0,Monthly,Home,Online,64,Scottish Borders,250.0,0.61,True,standard,1
3,GS-000001,POL-000001-01,2024-05-10,4,48,770.55,0.004,0,0,Monthly,Home,Online,64,Scottish Borders,250.0,0.61,True,standard,1
4,GS-000001,POL-000001-01,2025-05-10,5,60,791.13,0.031,0,0,Monthly,Home,Online,64,Scottish Borders,250.0,0.61,True,standard,1


In [4]:
# ============================================
# DROP IDENTIFIERS AND LEAKAGE
# ============================================
# - customer_id, policy_id: identifiers, no predictive value
# - decision_date, policy_year: time fields — tenure_months captures the same info
# - segment: ground truth label, would cause data leakage
# - region: too high-cardinality for this model iteration

drop_cols = [
    "customer_id",
    "policy_id",
    "decision_date",
    "policy_year",
    "segment",
    "region",
]

df = df.drop(columns=drop_cols)

print(f"Remaining columns: {len(df.columns)}")
print(list(df.columns))

Remaining columns: 13
['tenure_months', 'annual_premium', 'premium_change_pct', 'claims_last_year', 'total_claims', 'payment_frequency', 'policy_type', 'acquisition_channel', 'customer_age', 'excess_amount', 'no_claims_discount', 'last_year_renewed', 'will_renew']


In [5]:
# ============================================
# DERIVED FEATURES
# ============================================

# Tenure in years (more interpretable than months)
df["tenure_years"] = (df["tenure_months"] / 12).round(2)

# Claims per year of tenure — normalises claims by exposure
df["claims_per_year"] = (
    df["total_claims"] / df["tenure_years"].replace(0, np.nan)
).fillna(0).round(3)

# Flag: did the premium change more than 10% this year?
df["high_premium_change"] = (df["premium_change_pct"] > 0.10).astype(int)

# Flag: did the customer have any claims last year?
df["had_claim_last_year"] = (df["claims_last_year"] > 0).astype(int)

# Interaction: tenure × claim-free bonus
# A long-tenured customer with no recent claims is the "ideal" renewer
df["tenure_x_no_claim"] = (
    df["tenure_years"] * (1 - df["had_claim_last_year"])
).round(2)

print(f"After feature engineering: {len(df.columns)} columns")
print(list(df.columns))

After feature engineering: 18 columns
['tenure_months', 'annual_premium', 'premium_change_pct', 'claims_last_year', 'total_claims', 'payment_frequency', 'policy_type', 'acquisition_channel', 'customer_age', 'excess_amount', 'no_claims_discount', 'last_year_renewed', 'will_renew', 'tenure_years', 'claims_per_year', 'high_premium_change', 'had_claim_last_year', 'tenure_x_no_claim']


In [6]:
# ============================================
# ENCODE CATEGORICALS
# ============================================

# payment_frequency: binary
df["is_monthly_payer"] = (df["payment_frequency"] == "Monthly").astype(int)
df = df.drop(columns=["payment_frequency"])

# policy_type and acquisition_channel: one-hot
df = pd.get_dummies(
    df,
    columns=["policy_type", "acquisition_channel"],
    drop_first=True,
    dtype=int,
)

# last_year_renewed: bool → int
df["last_year_renewed"] = df["last_year_renewed"].astype(int)

print(f"After encoding: {len(df.columns)} columns")
print(list(df.columns))

After encoding: 23 columns
['tenure_months', 'annual_premium', 'premium_change_pct', 'claims_last_year', 'total_claims', 'customer_age', 'excess_amount', 'no_claims_discount', 'last_year_renewed', 'will_renew', 'tenure_years', 'claims_per_year', 'high_premium_change', 'had_claim_last_year', 'tenure_x_no_claim', 'is_monthly_payer', 'policy_type_Life', 'policy_type_Motor', 'policy_type_Pet', 'policy_type_Travel', 'acquisition_channel_Comparison Site', 'acquisition_channel_Direct', 'acquisition_channel_Online']


In [7]:
# ============================================
# FINAL MATRIX
# ============================================

# Move target to end for readability
target = df["will_renew"]
features = df.drop(columns=["will_renew"])

X = features.copy()
y = target.copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape:         {y.shape}")
print(f"\nTarget balance:")
print(y.value_counts(normalize=True).round(3))
print(f"\nFeature dtypes:")
print(X.dtypes.value_counts())
print(f"\nFirst 5 rows:")
X.head()

Feature matrix shape: (12199, 22)
Target shape:         (12199,)

Target balance:
will_renew
1    0.7
0    0.3
Name: proportion, dtype: float64

Feature dtypes:
int64      15
float64     7
Name: count, dtype: int64

First 5 rows:


,tenure_months,annual_premium,premium_change_pct,claims_last_year,total_claims,customer_age,excess_amount,no_claims_discount,last_year_renewed,tenure_years,...,had_claim_last_year,tenure_x_no_claim,is_monthly_payer,policy_type_Life,policy_type_Motor,policy_type_Pet,policy_type_Travel,acquisition_channel_Comparison Site,acquisition_channel_Direct,acquisition_channel_Online
0,12,792.25,0.033,0,0,64,250.0,0.61,1,1.0,...,0,1.0,1,0,0,0,0,0,0,1
1,24,793.53,0.034,0,0,64,250.0,0.61,1,2.0,...,0,2.0,1,0,0,0,0,0,0,1
2,36,824.42,0.074,0,0,64,250.0,0.61,1,3.0,...,0,3.0,1,0,0,0,0,0,0,1
3,48,770.55,0.004,0,0,64,250.0,0.61,1,4.0,...,0,4.0,1,0,0,0,0,0,0,1
4,60,791.13,0.031,0,0,64,250.0,0.61,1,5.0,...,0,5.0,1,0,0,0,0,0,0,1


In [8]:
# ============================================
# SANITY CHECKS
# ============================================

# No missing values
missing = X.isna().sum().sum()
print(f"Missing values in features: {missing}")

# No infinite values
infinite = np.isinf(X.select_dtypes(include=[np.number])).sum().sum()
print(f"Infinite values in features: {infinite}")

# All numeric
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns: {non_numeric}")

# Class balance
print(f"\nClass balance:")
print(f"  will_renew = 1: {(y == 1).sum():,} ({(y == 1).mean():.1%})")
print(f"  will_renew = 0: {(y == 0).sum():,} ({(y == 0).mean():.1%})")

Missing values in features: 0
Infinite values in features: 0
Non-numeric columns: []

Class balance:
  will_renew = 1: 8,543 (70.0%)
  will_renew = 0: 3,656 (30.0%)


In [9]:
# ============================================
# SAVE TO POSTGRES
# ============================================

# Recombine features + target for storage
features_out = X.copy()
features_out["will_renew"] = y.values

features_out.to_sql(
    name="features",
    con=engine,
    schema=DB_SCHEMA,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000,
)

with engine.connect() as conn:
    count = conn.execute(
        text(f"SELECT COUNT(*) FROM {DB_SCHEMA}.features;")
    ).scalar()

print(f"Wrote {count:,} rows to {DB_SCHEMA}.features")

Wrote 12,199 rows to retention.features


In [10]:
# ============================================
# SAVE TO CSV
# ============================================

features_out.to_csv(
    PROCESSED_DATA_DIR / "features_v1.csv",
    index=False,
)

print(f"Wrote {len(features_out):,} rows to data/processed/features_v1.csv")
print(f"File size: {(PROCESSED_DATA_DIR / 'features_v1.csv').stat().st_size / 1024:.1f} KB")

Wrote 12,199 rows to data/processed/features_v1.csv
File size: 843.0 KB


## Feature Summary

### Dropped
- `customer_id`, `policy_id` — identifiers
- `decision_date`, `policy_year` — captured by `tenure_months` / `tenure_years`
- `segment` — ground truth label (leakage)
- `region` — high cardinality

### Derived
- `tenure_years` — tenure in years (readable)
- `claims_per_year` — normalised claim frequency
- `high_premium_change` — binary flag for >10% premium rise
- `had_claim_last_year` — binary flag for any recent claim
- `tenure_x_no_claim` — interaction: long-tenured, claim-free customers

### Encoded
- `is_monthly_payer` — binary from `payment_frequency`
- `policy_type_*` — one-hot
- `acquisition_channel_*` — one-hot
- `last_year_renewed` — bool → int

### Output
- Saved to `retention.features` (Postgres)
- Saved to `data/processed/features_v1.csv` (GitHub)